 **Fetching Parameters & Creating Variables** 

In [0]:
#Catalog 
#catalog = "workspace"

#KEY Cols List 
#key_cols  = "['airport_id']"
#key_cols_list = eval(key_cols)

#CDC Cols 
#cdc_col = "modifiedDate"

#BCK- DATED REFRESH
#bckdated_refresh = ""

#Source Object 
#source_object = "silver_airports"

#source Schema
#source_schema = "silver"

#Target Schema 
#target_schema = "gold"

#Target Object
#target_object = "DimAirports"

#Surrogate Key Column
#surrogate_key = "DimAirportsKey"

In [0]:
#Catalog 
#catalog = "workspace"

#KEY Cols List 
#key_cols  = "['flight_id']"
#key_cols_list = eval(key_cols)

#CDC Cols 
#cdc_col = "modifiedDate"

#BCK- DATED REFRESH
#bckdated_refresh = ""

#Source Object 
#source_object = "silver_flights"

#source Schema
#source_schema = "silver"

#Target Schema 
#target_schema = "gold"

#Target Object
#target_object = "Dimflights"

#Surrogate Key Column
#surrogate_key = "DimflightsKey"

In [0]:
#Catalog 
catalog = "workspace"

#KEY Cols List 
key_cols  = "['passenger_id']"
key_cols_list = eval(key_cols)

#CDC Cols 
cdc_col = "modifiedDate"

#BCK- DATED REFRESH
bckdated_refresh = ""

#Source Object 
source_object = "silver_passengers"

#source Schema
source_schema = "silver"

#Target Schema 
target_schema = "gold"

#Target Object
target_object = "DimPassengers"

#Surrogate Key Column
surrogate_key = "DimPassengersKey"

##**Intermental Data Ingestion - Source Query**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
key_cols_list

['passenger_id']

In [0]:
# No Back Dated Refresh
if len(bckdated_refresh) == 0:
# If the table Exists in the Destination
    if spark.catalog.tableExists(f"workspace.{target_schema}.{target_object}"):

        last_load = spark.sql(f"select max({cdc_col}) from {target_schema}.{target_object}").collect()[0][0]
    else:
        last_load = "1900-01-01 00:00:00"

# Yes Back Dated Refresh
else:
    last_load = bckdated_refresh

   
#Test the last load
last_load

datetime.datetime(2026, 5, 4, 16, 48, 15, 422000)

In [0]:
df_src = spark.sql(f"select * from {source_schema}.{source_object} where {cdc_col} >= '{last_load}'")

In [0]:
df_src.display()

passenger_id,name,gender,nationality,modifiedDate
P0201,Michele Perry,Male,Czech Republic,2026-05-04T16:48:15.422Z
P0202,Javier Mccormick,Male,Bolivia,2026-05-04T16:48:15.422Z
P0203,Michelle Alvarado,Male,Iraq,2026-05-04T16:48:15.422Z
P0204,Jamie Harrell,Female,Norway,2026-05-04T16:48:15.422Z
P0205,James Bender,Female,Saint Barthelemy,2026-05-04T16:48:15.422Z
P0206,Amy Burns,Female,Malta,2026-05-04T16:48:15.422Z
P0207,Angela Sparks,Male,Nicaragua,2026-05-04T16:48:15.422Z
P0208,Cynthia Salas,Male,United States of America,2026-05-04T16:48:15.422Z
P0209,Christine Norman,Female,Sudan,2026-05-04T16:48:15.422Z
P0210,Robert Davis,Male,Puerto Rico,2026-05-04T16:48:15.422Z


**OLD vs NEW records -- for if cases - right now, we don't have the table, so it basically calls only the else statement**

In [0]:
spark.sql(f"SELECT '' AS flight_id, '' AS DimFlightsKey, 1900-01-01 AS create_date, 1900-01-01 AS update_date FROM workspace.silver.silver_flights").display()

flight_id,DimFlightsKey,create_date,update_date
,,1898,1898
,,1898,1898
,,1898,1898
,,1898,1898
,,1898,1898
,,1898,1898
,,1898,1898
,,1898,1898
,,1898,1898
,,1898,1898


In [0]:
if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):

    #Key Column String for Incremental
    key_cols_string_incremental = ", ".join(key_cols_list)

    df_trg = spark.sql(f"SELECT {key_cols_string_incremental}, {surrogate_key}, create_date, update_date FROM {catalog}.{target_schema}.{target_object}")
else:

    #Key Columns String for Initial
    key_cols_string_init = [f"'' AS {i}" for i in key_cols_list]
    key_cols_string_init = ", ".join(key_cols_string_init)
    key_cols_string_init
    df_trg = spark.sql(f"SELECT {key_cols_string_init}, CAST ('0'AS INT) AS {surrogate_key}, CAST('1900-01-01 00:00:00' AS timestamp) AS create_date, CAST('1900-01-01 00:00:00' AS timestamp) AS update_date WHERE 1=0""")

In [0]:
df_trg.display()



passenger_id,DimPassengersKey,create_date,update_date
P0201,1,2026-05-10T19:13:33.317Z,2026-05-10T19:13:33.317Z
P0202,2,2026-05-10T19:13:33.317Z,2026-05-10T19:13:33.317Z
P0203,3,2026-05-10T19:13:33.317Z,2026-05-10T19:13:33.317Z
P0204,4,2026-05-10T19:13:33.317Z,2026-05-10T19:13:33.317Z
P0205,5,2026-05-10T19:13:33.317Z,2026-05-10T19:13:33.317Z
P0206,6,2026-05-10T19:13:33.317Z,2026-05-10T19:13:33.317Z
P0207,7,2026-05-10T19:13:33.317Z,2026-05-10T19:13:33.317Z
P0208,8,2026-05-10T19:13:33.317Z,2026-05-10T19:13:33.317Z
P0209,9,2026-05-10T19:13:33.317Z,2026-05-10T19:13:33.317Z
P0210,10,2026-05-10T19:13:33.317Z,2026-05-10T19:13:33.317Z


##**Join Condition**

In [0]:
join_condition = ' AND '.join([f"src{i} = trg{i}" for i in key_cols_list])

In [0]:
df_src.createOrReplaceTempView("src")
df_trg.createOrReplaceTempView("trg")

join_condition = " AND ".join([f"src.{col} = trg.{col}" for col in key_cols_list])

df_join = spark.sql(f"""
    SELECT src.*,
           trg.{surrogate_key},
           trg.create_date,
           trg.update_date
    FROM src
    LEFT JOIN trg
    ON {join_condition}
""")

In [0]:
df_join.display()

passenger_id,name,gender,nationality,modifiedDate,DimPassengersKey,create_date,update_date
P0201,Michele Perry,Male,Czech Republic,2026-05-04T16:48:15.422Z,1,2026-05-10T19:13:33.317Z,2026-05-10T19:13:33.317Z
P0202,Javier Mccormick,Male,Bolivia,2026-05-04T16:48:15.422Z,2,2026-05-10T19:13:33.317Z,2026-05-10T19:13:33.317Z
P0203,Michelle Alvarado,Male,Iraq,2026-05-04T16:48:15.422Z,3,2026-05-10T19:13:33.317Z,2026-05-10T19:13:33.317Z
P0204,Jamie Harrell,Female,Norway,2026-05-04T16:48:15.422Z,4,2026-05-10T19:13:33.317Z,2026-05-10T19:13:33.317Z
P0205,James Bender,Female,Saint Barthelemy,2026-05-04T16:48:15.422Z,5,2026-05-10T19:13:33.317Z,2026-05-10T19:13:33.317Z
P0206,Amy Burns,Female,Malta,2026-05-04T16:48:15.422Z,6,2026-05-10T19:13:33.317Z,2026-05-10T19:13:33.317Z
P0207,Angela Sparks,Male,Nicaragua,2026-05-04T16:48:15.422Z,7,2026-05-10T19:13:33.317Z,2026-05-10T19:13:33.317Z
P0208,Cynthia Salas,Male,United States of America,2026-05-04T16:48:15.422Z,8,2026-05-10T19:13:33.317Z,2026-05-10T19:13:33.317Z
P0209,Christine Norman,Female,Sudan,2026-05-04T16:48:15.422Z,9,2026-05-10T19:13:33.317Z,2026-05-10T19:13:33.317Z
P0210,Robert Davis,Male,Puerto Rico,2026-05-04T16:48:15.422Z,10,2026-05-10T19:13:33.317Z,2026-05-10T19:13:33.317Z


In [0]:
#OLD Records
df_old = df_join.filter(col(f'{surrogate_key}').isNotNull())

#NEW Records
df_new = df_join.filter(col(f'{surrogate_key}').isNull())



In [0]:
df_old.display()
df_new.display()

passenger_id,name,gender,nationality,modifiedDate,DimPassengersKey,create_date,update_date
P0201,Michele Perry,Male,Czech Republic,2026-05-04T16:48:15.422Z,1,2026-05-10T19:13:33.317Z,2026-05-10T19:13:33.317Z
P0202,Javier Mccormick,Male,Bolivia,2026-05-04T16:48:15.422Z,2,2026-05-10T19:13:33.317Z,2026-05-10T19:13:33.317Z
P0203,Michelle Alvarado,Male,Iraq,2026-05-04T16:48:15.422Z,3,2026-05-10T19:13:33.317Z,2026-05-10T19:13:33.317Z
P0204,Jamie Harrell,Female,Norway,2026-05-04T16:48:15.422Z,4,2026-05-10T19:13:33.317Z,2026-05-10T19:13:33.317Z
P0205,James Bender,Female,Saint Barthelemy,2026-05-04T16:48:15.422Z,5,2026-05-10T19:13:33.317Z,2026-05-10T19:13:33.317Z
P0206,Amy Burns,Female,Malta,2026-05-04T16:48:15.422Z,6,2026-05-10T19:13:33.317Z,2026-05-10T19:13:33.317Z
P0207,Angela Sparks,Male,Nicaragua,2026-05-04T16:48:15.422Z,7,2026-05-10T19:13:33.317Z,2026-05-10T19:13:33.317Z
P0208,Cynthia Salas,Male,United States of America,2026-05-04T16:48:15.422Z,8,2026-05-10T19:13:33.317Z,2026-05-10T19:13:33.317Z
P0209,Christine Norman,Female,Sudan,2026-05-04T16:48:15.422Z,9,2026-05-10T19:13:33.317Z,2026-05-10T19:13:33.317Z
P0210,Robert Davis,Male,Puerto Rico,2026-05-04T16:48:15.422Z,10,2026-05-10T19:13:33.317Z,2026-05-10T19:13:33.317Z


passenger_id,name,gender,nationality,modifiedDate,DimPassengersKey,create_date,update_date
P0221,Amy Welch,Male,Croatia,2026-05-10T19:42:53.090Z,null,null,null
P0222,Maria Taylor,Male,Lao People's Democratic Republic,2026-05-10T19:42:53.090Z,null,null,null
P0223,Nicholas Gomez,Female,Cook Islands,2026-05-10T19:42:53.090Z,null,null,null
P0224,Jason Jensen,Female,Rwanda,2026-05-10T19:42:53.090Z,null,null,null
P0225,William Lopez,Male,Heard Island and McDonald Islands,2026-05-10T19:42:53.090Z,null,null,null


**Preparing DF_OLD**

In [0]:
df_old_enr = df_old.withColumn('update_date', current_timestamp())

**Preparing DF_new**

In [0]:
from pyspark.sql.functions import col

if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
    max_surrogate_key = spark.sql(f"""
        SELECT max(CAST({surrogate_key} AS BIGINT)) FROM {catalog}.{target_schema}.{target_object}
    """).collect()[0][0]
    
    if max_surrogate_key is None:
        max_surrogate_key = 0
    
    df_new_enr = df_new.withColumn(f'{surrogate_key}', (lit(max_surrogate_key)+lit(1)+monotonically_increasing_id()).cast("bigint"))\
        .withColumn('create_date', current_timestamp())\
        .withColumn('update_date', current_timestamp())
else:
    max_surrogate_key = 0
    df_new_enr = df_new.withColumn(f'{surrogate_key}', (lit(max_surrogate_key)+lit(1)+monotonically_increasing_id()).cast("bigint"))\
        .withColumn('create_date', current_timestamp())\
        .withColumn('update_date', current_timestamp())

In [0]:
max_surrogate_key

220

In [0]:
df_old_enr.display()

passenger_id,name,gender,nationality,modifiedDate,DimPassengersKey,create_date,update_date
P0201,Michele Perry,Male,Czech Republic,2026-05-04T16:48:15.422Z,1,2026-05-10T19:13:33.317Z,2026-05-10T19:52:07.547Z
P0202,Javier Mccormick,Male,Bolivia,2026-05-04T16:48:15.422Z,2,2026-05-10T19:13:33.317Z,2026-05-10T19:52:07.547Z
P0203,Michelle Alvarado,Male,Iraq,2026-05-04T16:48:15.422Z,3,2026-05-10T19:13:33.317Z,2026-05-10T19:52:07.547Z
P0204,Jamie Harrell,Female,Norway,2026-05-04T16:48:15.422Z,4,2026-05-10T19:13:33.317Z,2026-05-10T19:52:07.547Z
P0205,James Bender,Female,Saint Barthelemy,2026-05-04T16:48:15.422Z,5,2026-05-10T19:13:33.317Z,2026-05-10T19:52:07.547Z
P0206,Amy Burns,Female,Malta,2026-05-04T16:48:15.422Z,6,2026-05-10T19:13:33.317Z,2026-05-10T19:52:07.547Z
P0207,Angela Sparks,Male,Nicaragua,2026-05-04T16:48:15.422Z,7,2026-05-10T19:13:33.317Z,2026-05-10T19:52:07.547Z
P0208,Cynthia Salas,Male,United States of America,2026-05-04T16:48:15.422Z,8,2026-05-10T19:13:33.317Z,2026-05-10T19:52:07.547Z
P0209,Christine Norman,Female,Sudan,2026-05-04T16:48:15.422Z,9,2026-05-10T19:13:33.317Z,2026-05-10T19:52:07.547Z
P0210,Robert Davis,Male,Puerto Rico,2026-05-04T16:48:15.422Z,10,2026-05-10T19:13:33.317Z,2026-05-10T19:52:07.547Z


** Unioning OLD and New Records**

In [0]:
df_union = df_old_enr.unionByName(df_new_enr)

In [0]:
df_old.printSchema()

root
 |-- passenger_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- nationality: string (nullable = true)
 |-- modifiedDate: timestamp (nullable = true)
 |-- DimPassengersKey: long (nullable = true)
 |-- create_date: timestamp (nullable = true)
 |-- update_date: timestamp (nullable = true)



In [0]:
df_union.display()

passenger_id,name,gender,nationality,modifiedDate,DimPassengersKey,create_date,update_date
P0201,Michele Perry,Male,Czech Republic,2026-05-04T16:48:15.422Z,1,2026-05-10T19:13:33.317Z,2026-05-10T19:52:08.934Z
P0202,Javier Mccormick,Male,Bolivia,2026-05-04T16:48:15.422Z,2,2026-05-10T19:13:33.317Z,2026-05-10T19:52:08.934Z
P0203,Michelle Alvarado,Male,Iraq,2026-05-04T16:48:15.422Z,3,2026-05-10T19:13:33.317Z,2026-05-10T19:52:08.934Z
P0204,Jamie Harrell,Female,Norway,2026-05-04T16:48:15.422Z,4,2026-05-10T19:13:33.317Z,2026-05-10T19:52:08.934Z
P0205,James Bender,Female,Saint Barthelemy,2026-05-04T16:48:15.422Z,5,2026-05-10T19:13:33.317Z,2026-05-10T19:52:08.934Z
P0206,Amy Burns,Female,Malta,2026-05-04T16:48:15.422Z,6,2026-05-10T19:13:33.317Z,2026-05-10T19:52:08.934Z
P0207,Angela Sparks,Male,Nicaragua,2026-05-04T16:48:15.422Z,7,2026-05-10T19:13:33.317Z,2026-05-10T19:52:08.934Z
P0208,Cynthia Salas,Male,United States of America,2026-05-04T16:48:15.422Z,8,2026-05-10T19:13:33.317Z,2026-05-10T19:52:08.934Z
P0209,Christine Norman,Female,Sudan,2026-05-04T16:48:15.422Z,9,2026-05-10T19:13:33.317Z,2026-05-10T19:52:08.934Z
P0210,Robert Davis,Male,Puerto Rico,2026-05-04T16:48:15.422Z,10,2026-05-10T19:13:33.317Z,2026-05-10T19:52:08.934Z


In [0]:
# Drop the surrogate_key from df_old since it's NULL anyway on first run
df_old_clean = df_old.drop(f'{surrogate_key}').drop('create_date').drop('update_date')

In [0]:
# Union should work now
df_union = df_new_enr.unionByName(df_old_clean, allowMissingColumns=True)

df_union.display()

passenger_id,name,gender,nationality,modifiedDate,DimPassengersKey,create_date,update_date
P0221,Amy Welch,Male,Croatia,2026-05-10T19:42:53.090Z,8589934813,2026-05-10T19:52:10.440Z,2026-05-10T19:52:10.440Z
P0222,Maria Taylor,Male,Lao People's Democratic Republic,2026-05-10T19:42:53.090Z,8589934814,2026-05-10T19:52:10.440Z,2026-05-10T19:52:10.440Z
P0223,Nicholas Gomez,Female,Cook Islands,2026-05-10T19:42:53.090Z,8589934815,2026-05-10T19:52:10.440Z,2026-05-10T19:52:10.440Z
P0224,Jason Jensen,Female,Rwanda,2026-05-10T19:42:53.090Z,8589934816,2026-05-10T19:52:10.440Z,2026-05-10T19:52:10.440Z
P0225,William Lopez,Male,Heard Island and McDonald Islands,2026-05-10T19:42:53.090Z,8589934817,2026-05-10T19:52:10.440Z,2026-05-10T19:52:10.440Z
P0201,Michele Perry,Male,Czech Republic,2026-05-04T16:48:15.422Z,null,null,null
P0202,Javier Mccormick,Male,Bolivia,2026-05-04T16:48:15.422Z,null,null,null
P0203,Michelle Alvarado,Male,Iraq,2026-05-04T16:48:15.422Z,null,null,null
P0204,Jamie Harrell,Female,Norway,2026-05-04T16:48:15.422Z,null,null,null
P0205,James Bender,Female,Saint Barthelemy,2026-05-04T16:48:15.422Z,null,null,null


## **UPSERT**

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):

    dlt_obj = DeltaTable.forName(spark, f"{catalog}.{target_schema}.{target_object}")
    dlt_obj.alias("trg").merge(df_union.alias("src"), f"trg.{surrogate_key} = src.{surrogate_key}")\
                        .whenMatchedUpdateAll(condition = f"src.{cdc_col} >= trg.{cdc_col}")\
                        .whenNotMatchedInsertAll()\
                        .execute()
    

else:
    df_union.write.format("delta")\
        .mode("append")\
        .saveAsTable(f"{catalog}.{target_schema}.{target_object}")